# QUESTION 1 : Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
spark.version

'4.1.1'

# QUESTION 2 : Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe. Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

a. 6MB <br>
b. 25MB <br>
c. 75MB <br>
d. 100MB <br>

In [4]:
# Download the dataset of yellow taxi data on November 2025
import wget
wget.download('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet')

100% [........................................................................] 71134255 / 71134255

'yellow_tripdata_2025-11.parquet'

In [5]:
#check the size using linux command ls -lh
!ls -lh yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 HP 197121 68M Mar  6 11:34 yellow_tripdata_2025-11.parquet


In [6]:
!ls -l yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 HP 197121 71134255 Mar  6 11:34 yellow_tripdata_2025-11.parquet


Using ls -lh command give a result of 68MB, and using ls -l command give a result of 71.134.255 bytes. The closest multiple chouce answer is 75 MB, so <b> the answer is (c) </b>

# QUESTION 3 : COUNT RECORDS

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

a. 62,610 <br>
b. 102,340 <br>
c. 162,604 <br>
d. 225,768 <br>

In [12]:
#create dataframe for yellow taxi data
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \ #this option will create schema that refer to the initial column
    .parquet('yellow_tripdata_2025-11.parquet')

In [13]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [16]:
#Add pickup_datetime column with to_date function 
from pyspark.sql import functions as F

#counting the filtered data with spark dataframe method
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

In [25]:
df = df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime))

In [26]:
df.registerTempTable('yellow_2025_11')

In [27]:
spark.sql("""
SELECT COUNT(1) from yellow_2025_11
""").show()

+--------+
|count(1)|
+--------+
| 4181444|
+--------+



In [28]:
#counting the filtered data with spark sql method
spark.sql("""
SELECT COUNT(1) from yellow_2025_11
WHERE to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [29]:
#counting the filtered data with spark sql method
spark.sql("""
SELECT COUNT(1) from yellow_2025_11
WHERE pickup_date = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



From executing df.filter("pickup_date = '2025-11-15'").count() and executing spark.sql, we conclude that <b> the answer is 162604 (c) </b>

# QUESTION 4 : LONGEST TRIP

What is the length of the longest trip in the dataset (in hours)? <br>
a. 22.7 <br>
b. 58.2 <br>
c. 90.6 <br>
d. 134.5

In [35]:
#use Pandas dataframe to make the display cleaner to see
import pandas as pd
pd.set_option('display.max_columns', None)

df.limit(5).toPandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_date
0,7,2025-11-01 00:13:25,2025-11-01 00:13:25,1,1.68,1,N,43,186,1,14.9,0.00,0.5,1.50,0.00,1.0,22.15,2.5,0.00,0.75,2025-11-01
1,2,2025-11-01 00:49:07,2025-11-01 01:01:22,1,2.28,1,N,142,237,1,14.2,1.00,0.5,4.99,0.00,1.0,24.94,2.5,0.00,0.75,2025-11-01
2,1,2025-11-01 00:07:19,2025-11-01 00:20:41,0,2.70,1,N,163,238,1,15.6,4.25,0.5,4.27,0.00,1.0,25.62,2.5,0.00,0.75,2025-11-01
3,2,2025-11-01 00:00:00,2025-11-01 01:01:03,3,12.87,1,N,138,261,1,66.7,6.00,0.5,0.00,6.94,1.0,86.14,2.5,1.75,0.75,2025-11-01
4,1,2025-11-01 00:18:50,2025-11-01 00:49:32,0,8.40,1,N,138,37,2,39.4,7.75,0.5,0.00,0.00,1.0,48.65,0.0,1.75,0.00,2025-11-01


In [38]:
spark.sql("""
SELECT
    MAX(TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime)) / 60 AS max_duration_hours
FROM yellow_2025_11
""").show()

+------------------+
|max_duration_hours|
+------------------+
| 90.63333333333334|
+------------------+



<b> TIMESTAMPDIFF </b> function will substract the tpep_dropoff_datetime with tpep_pickup_datetime, to know the difference between two time. <br>
<b> MAX </b> function will find the data with the biggest difference

from the result, <b> the answer is 90.6 (c) </b>

# QUESTION 5 : SPARK UI

Spark's User Interface, which shows the application's dashboard, runs on which local port? <br>
a. 80 <br>
b. 443 <br>
c. 4040 <br>
d. 8080

The Default port for Spark's User Interface is port 4040 <br>
You can confirm it, by go to this address: http://localhost:4040/jobs/ <br>

so the <b> answer is (c) </b>

# QUESTION 6 : Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark: <br>
```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone? <br>

a. Governor's Island/Ellis Island/Liberty Island <br>
b. Arden Heights <br>
c. Rikers Island <br>
d. Jamaica Bay <br>

If multiple answers are correct, select any

In [44]:
#downloading the zone lookup data
import wget
wget.download('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')

df_zone = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('taxi_zone_lookup.csv')

df_zone.limit(5).toPandas()

100% [..........................................................] 12331 / 12331

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [45]:
#viewing the tripdata table to compare data
df.limit(5).toPandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_date
0,7,2025-11-01 00:13:25,2025-11-01 00:13:25,1,1.68,1,N,43,186,1,14.9,0.00,0.5,1.50,0.00,1.0,22.15,2.5,0.00,0.75,2025-11-01
1,2,2025-11-01 00:49:07,2025-11-01 01:01:22,1,2.28,1,N,142,237,1,14.2,1.00,0.5,4.99,0.00,1.0,24.94,2.5,0.00,0.75,2025-11-01
2,1,2025-11-01 00:07:19,2025-11-01 00:20:41,0,2.70,1,N,163,238,1,15.6,4.25,0.5,4.27,0.00,1.0,25.62,2.5,0.00,0.75,2025-11-01
3,2,2025-11-01 00:00:00,2025-11-01 01:01:03,3,12.87,1,N,138,261,1,66.7,6.00,0.5,0.00,6.94,1.0,86.14,2.5,1.75,0.75,2025-11-01
4,1,2025-11-01 00:18:50,2025-11-01 00:49:32,0,8.40,1,N,138,37,2,39.4,7.75,0.5,0.00,0.00,1.0,48.65,0.0,1.75,0.00,2025-11-01


In [46]:
df_zone.registerTempTable('zone_data')

C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [53]:
# identifying the least frequent pickup location by joining df and df_zone table, on df.PULocationID = df_zone.locationID
spark.sql ("""
    SELECT dfz.LocationID, dfz.Zone, count(1)
    FROM yellow_2025_11 df
    JOIN zone_data dfz ON df.PULocationID = dfz.locationID
    GROUP BY dfz.locationID, dfz.Zone
    ORDER BY count(1)
""").limit(5).toPandas()

,LocationID,Zone,count(1)
0,105,Governor's Island/Ellis Island/Liberty Island,1
1,5,Arden Heights,1
2,84,Eltingville/Annadale/Prince's Bay,1
3,187,Port Richmond,3
4,204,Rossville/Woodrow,4


There are 3 places with count of pickup location is equal to 1, which are:
- Governor's Island/Ellis Island/Liberty Island	
- Eltingville/Annadale/Prince's Bay	
- Arden Heights

comparing to the multiple choice, <b> the answer could be (a) or (b) </b>